# 23b · Set 7b: Qwen 7B learns to shortlist, trained on 8,000 questions (LoRA, A100)

Notebook 22 trained Qwen 3B on about 4,000 questions: it then found both gold passages on 84.8% of the test questions (Qwen 7B
untrained: 79.5%), and with Qwen 32B answering it beat the cheap fixed setup by about 4 points at 42% less compute. Its
validation curve was still rising. This notebook trains **Qwen 7B** on **8,000** questions, with the same recipe.

Notebooks 23a (Qwen 3B, 12,000 questions) and 23b (Qwen 7B, 8,000 questions) are independent: run them in two Colab sessions at
the same time if you can. They share the training questions (whichever starts first builds them; about 10 minutes, once).

| | |
|---|---|
| Same as notebook 22 | Input (question + 20 passages as text), target ("Selected references: [a] [b]"), LoRA r = 16, learning rate, test questions, answerers |
| Training questions | 8,000 of the 12,000 hard-level questions built from HotpotQA's training file (all 15,661 hard questions form the distractor corpus) |
| Validation | 300 further training-file questions, scored after 2,000, 4,000, 8,000 questions; the best checkpoint is kept |
| Faster training | 4 questions per GPU pass, and the loss is computed only on the answer tokens (same result, less memory and time) |
| Resumable | A resume point is saved every 1,000 questions; if the session stops, run the notebook again |

**My estimate:** training about 2.5–4.5 hours; about 3–5 hours in total with the retrieval, the answers and the model copies. These are
estimates (notebook 22's 4,000 questions took 67 minutes on the 3B model, one question per pass).

Output: `runs/Set_3__self-learnt-retrieval/<timestamp>_<GPU>_ft7_8k/` with `adapter/` (training curve, checkpoints) and `results/summary.md`.

**Runtime → Change runtime type → A100 GPU.**

In [ ]:
# ==== settings ====
PROJECT_DIR = "/content/drive/MyDrive/0_EngD_assignments/0_Small_Big_Model_Switching"
EXPERIMENT  = "Set_3__self-learnt-retrieval"
EXP_SET5    = "Set_2__question-reading-routers"
OLD_SWEEP_EXPERIMENT = "Set_1__3-modality-size"
FRESH_DATA  = f"{PROJECT_DIR}/data/hotpotqa_q/fresh200_c20_s43"
MODEL   = "q7"                   # the retriever to train: Qwen 7B
N_TRAIN = 8000                      # training questions used (of the 12,000 built)
N_VAL   = 300                         # validation questions (checkpoint choice only)
CKPT_QUESTIONS = [2000, 4000, 8000]     # validation checkpoints, in questions seen
TIME_LIMIT_MIN = 600                  # stops cleanly at this limit; run the notebook again to continue
NEW_RUN = False                       # False = continue the latest run of this notebook
DRY_RUN = False                       # True = tiny model and fake answers on CPU: tests the whole notebook in a few minutes

EXPECTED_CODE = {
 "ssw.py": "c9fbdc7f6b91c58b607021b8be8c03ae2ff67dc1024fd2e24fffeb18e239ed98",
 "sweep.py": "9382292c6fe396cf79c73b212aaf6c2d677cae7f9311c3a5304f0578b52e6bb6",
 "routers.py": "8a293df1c8955716950f535cf9a3ced24439e00250173da1a3df110c0ed87304",
 "hidden.py": "574e6c1390e47c6de833a0c10b32c24eed0dee73a35eabc17f58842835e7f105",
 "final.py": "f18a6fba3d1bdd1f6700f6dc535f10ec0c41dbf5192b7b39ad097f8a1b88be53",
 "retriever_check.py": "fde08c549ecbd6bfc5527a42c731aa51dcff687a64818d68cf9cb30090e017e3",
 "ensemble.py": "5c5e87175d2484d806fda913a9555f71378a6031f050c37e89c20305942b76fc",
 "experts.py": "17ed39957e54853d178b47fa73ba328b50f0589b129ccb191a2efb0a2dea466b",
 "retft.py": "4a0d27d24459ba179c60dcce39a85285ae422dfa9e8d7cbc160e79a0a9932879"
}

In [ ]:
import subprocess, sys, os, json, shutil, time
gpu = subprocess.run("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader 2>/dev/null",
                     shell=True, capture_output=True, text=True).stdout.strip()
print("GPU:", gpu or "none")
if not DRY_RUN:
    assert gpu, "No GPU: Runtime -> Change runtime type -> A100 GPU (or set DRY_RUN = True)."
    if "T4" in gpu:
        raise SystemExit("This runtime has a T4, which cannot run this: the 7B model alone (15 GB) does not fit in its memory.\n"
                         "Runtime -> Change runtime type -> A100 GPU (or G4 / L4) -> Save, then run all cells again.")
    if "A100" not in gpu and "G4" not in gpu and "RTX PRO 6000" not in gpu:
        print("Note: planned for an A100 or G4; this GPU works but timings will differ.")
GPU_TAG = "cpu" if not gpu else gpu.split(",")[0].replace("NVIDIA", "").replace("Tesla", "").strip().split()[0].split("-")[0]
T_START = time.time()

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

### Check that Drive is serving the current code; find the earlier runs

In [ ]:
import glob, hashlib
from datetime import datetime
from zoneinfo import ZoneInfo
def sha(p):
    return hashlib.sha256(open(p, "rb").read()).hexdigest()
CODE_DIR = "/content/set7code"
os.makedirs(CODE_DIR, exist_ok=True)
stale = []
for f, want in EXPECTED_CODE.items():
    shutil.copy(f"{PROJECT_DIR}/code/{f}", f"{CODE_DIR}/{f}")
    if sha(f"{CODE_DIR}/{f}") != want:
        stale.append(f)
assert not stale, (f"Drive is serving a different version of {stale} than this notebook was made for. Wait two minutes, then "
                   "Runtime -> Disconnect and delete runtime, and run again.")
sweeps = [d for d in sorted(glob.glob(f"{PROJECT_DIR}/runs/{OLD_SWEEP_EXPERIMENT}/*")) if "_dryrun" not in d and os.path.exists(f"{d}/results/per_question.jsonl")
          and os.path.exists(f"{d}/sweep_status.json") and json.load(open(f"{d}/sweep_status.json")).get("complete")]
fresh = [d for d in sorted(glob.glob(f"{PROJECT_DIR}/runs/{EXP_SET5}/*_final")) if os.path.exists(f"{d}/step/retrieval_q7_c1_p3.jsonl")]
set6 = [d for d in sorted(glob.glob(f"{PROJECT_DIR}/runs/{EXP_SET5}/*_set6")) if os.path.exists(f"{d}/step/arms_q32.jsonl")]
set7 = [d for d in sorted(glob.glob(f"{PROJECT_DIR}/runs/{EXPERIMENT}/*_ft*")) if "_dryrun" not in d and os.path.exists(f"{d}/step/answers_q32.jsonl")]
assert sweeps and fresh, "Needs the finished runs of notebooks 11 and 17."
OLD_SWEEP, FRESH_RUN = sweeps[-1], fresh[-1]
CACHE_RUN = set6[-1] if set6 else ""
OLD_DATA = json.load(open(f"{OLD_SWEEP}/config.json"))["data_dir"]
TRAIN_DATA = f"{PROJECT_DIR}/data/hotpotqa_train/train12000_val300_c20_hard_all_s7" + ("_dryrun" if DRY_RUN else "")
SESSION = datetime.now(ZoneInfo("Asia/Singapore")).strftime("%Y%m%d-%H%M%S")
tag = "_ft7_8k_dryrun" if DRY_RUN else "_ft7_8k"
prev = sorted(glob.glob(f"{PROJECT_DIR}/runs/{EXPERIMENT}/*{tag}"))
RUN_DIR = prev[-1] if (prev and not NEW_RUN) else f"{PROJECT_DIR}/runs/{EXPERIMENT}/{SESSION}_{GPU_TAG}{tag}"
CACHE_ANSWERS = [d for d in set7 if d != RUN_DIR]
SESSION_DIR = f"{RUN_DIR}/sessions/{SESSION}"
for d in (f"{RUN_DIR}/code", f"{RUN_DIR}/logs", f"{RUN_DIR}/step", SESSION_DIR):
    os.makedirs(d, exist_ok=True)
for f in EXPECTED_CODE:
    shutil.copy(f"{CODE_DIR}/{f}", f"{RUN_DIR}/code/{f}")
TIMES = {}
print("Code files: current.")
print("Old questions:   ", OLD_SWEEP)
print("Fresh questions: ", FRESH_RUN)
print("Answers to reuse:", [CACHE_RUN] + CACHE_ANSWERS)
print("Training data:   ", TRAIN_DATA)
print("Run folder:      ", RUN_DIR)

In [ ]:
MODELS = {   # key: model, folder under models/, parameters (for compute), nominal size (for the AgentTTS budget formula)
    "q3":  {"id": "Qwen/Qwen2.5-3B-Instruct",        "dir": "Qwen2.5-3B-Instruct",        "params": 3.09e9,  "nominal": 3e9},
    "q7":  {"id": "Qwen/Qwen2.5-7B-Instruct",        "dir": "Qwen2.5-7B-Instruct",        "params": 7.62e9,  "nominal": 7e9},
    "q32": {"id": "Qwen/Qwen2.5-32B-Instruct-AWQ",   "dir": "Qwen2.5-32B-Instruct-AWQ",   "params": 32.76e9, "nominal": 32e9},
    "l3":  {"id": "meta-llama/Llama-3.2-3B-Instruct", "dir": "Llama-3.2-3B-Instruct",     "params": 3.21e9,  "nominal": 3e9},
    "l8":  {"id": "meta-llama/Llama-3.1-8B-Instruct", "dir": "Llama-3.1-8B-Instruct",     "params": 8.03e9,  "nominal": 8e9},
}
LOCAL = "/content/models"
for m in MODELS.values():
    m["path"] = f"{LOCAL}/{m['dir']}"

MODELS = {k: v for k, v in MODELS.items() if k in ("q3", "q7", "l8", "q32")}
FT_KEY, FT_NAME = "q7ft8k", "ft7_8k"
FT_LABEL = f"Qwen 7B, fine-tuned on {N_TRAIN:,} questions (direct prompt)"
MODELS[FT_KEY] = dict(MODELS[MODEL], id=f"{MODELS[MODEL]['id']} + LoRA trained in this run", dir=FT_KEY + "_merged", path=f"{LOCAL}/{FT_KEY}_merged")
json.dump(MODELS, open(f"{RUN_DIR}/models.json", "w"), indent=1)
json.dump({"experiment": EXPERIMENT, "set": f"7b retriever fine-tuning ({MODEL}, {N_TRAIN} questions)", "old_sweep": OLD_SWEEP, "fresh_run": FRESH_RUN,
           "cache": [CACHE_RUN] + CACHE_ANSWERS, "old_data": OLD_DATA, "fresh_data": FRESH_DATA, "train_data": TRAIN_DATA, "n_train": N_TRAIN, "n_val": N_VAL,
           "gpu": gpu, "dry_run": DRY_RUN, "code": EXPECTED_CODE, "started": SESSION}, open(f"{RUN_DIR}/config.json", "w"), indent=1)

def minutes_left():
    return max(0.0, TIME_LIMIT_MIN - (time.time() - T_START) / 60)

def run_stage(args, log_name):
    cmd = [sys.executable, "-u", f"{CODE_DIR}/retft.py"] + [str(a) for a in args]
    t0 = time.time()
    with open(f"{RUN_DIR}/logs/{log_name}", "a") as log, open(f"{SESSION_DIR}/{log_name}", "a") as slog:
        p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            log.write(line); slog.write(line)
            if not any(k in line for k in ("it/s]", "s/it]", "Loading checkpoint", "INFO ", "(EngineCore", "WARNING ", "Warning", "warn", "Loading weights", "Writing model")):
                print(line, end="")
        p.wait()
    TIMES[log_name.replace(".log", "")] = TIMES.get(log_name.replace(".log", ""), 0) + round(time.time() - t0)
    json.dump(TIMES, open(f"{SESSION_DIR}/stage_times.json", "w"), indent=1)
    print(f"--- {log_name}: exit {p.returncode}, {time.time() - t0:.0f}s, {minutes_left():.0f} min left of the limit")
    assert p.returncode == 0, f"stage failed - see {RUN_DIR}/logs/{log_name}"

ADAPTER = f"{RUN_DIR}/adapter"
def trained():
    s = f"{ADAPTER}/train_state.json"
    return os.path.exists(s) and "chosen" in json.load(open(s))
def merged():
    return os.path.exists(f"{MODELS[FT_KEY]['path']}/config.json")
print("training data built:", os.path.exists(f"{TRAIN_DATA}/val.json"), "| trained:", trained())

### 1. Build the training questions (once, shared by 23a and 23b; kept on Drive)

In [ ]:
if not os.path.exists(f"{TRAIN_DATA}/val.json"):
    if DRY_RUN:     # stand-in for the training file: the dev file itself (only to test the code; never used for real)
        run_stage(["build_train", "--hotpot_train", f"{PROJECT_DIR}/AgentTTS/data/HotpotQA/hotpotqa-dev.json", "--out_dir", TRAIN_DATA,
                   "--n_train", 80, "--n_val", 12, "--pool", 1500], "build_train.log")
    else:
        run_stage(["build_train", "--out_dir", TRAIN_DATA, "--n_train", 12000, "--n_val", 300, "--pool", 0], "build_train.log")
else:
    print("training questions already built:", TRAIN_DATA)
print(json.load(open(f"{TRAIN_DATA}/data_config.json")))

### 2. Copy the models (Llama 8B and Qwen 32B in the background), install peft, train the LoRA adapter on Qwen 7B

In [ ]:
import threading
COPY = {"err": None, "done": set(), "seconds": {}}
def _copy_all(keys):
    try:
        for key in keys:
            m = MODELS[key]
            src = f"{PROJECT_DIR}/models/{m['dir']}"
            if not os.path.exists(f"{m['path']}/config.json"):
                assert os.path.exists(f"{src}/config.json"), f"{src} missing - run notebook 10 first."
                t = time.time()
                shutil.rmtree(m["path"] + ".part", ignore_errors=True)
                shutil.copytree(src, m["path"] + ".part")
                os.rename(m["path"] + ".part", m["path"])
                COPY["seconds"][key] = round(time.time() - t)
            COPY["done"].add(key)
    except Exception as e:
        COPY["err"] = e

def wait_local(key):
    if DRY_RUN:
        return
    t = time.time()
    while key not in COPY["done"]:
        if COPY["err"]:
            raise COPY["err"]
        time.sleep(5)
    if time.time() - t > 10:
        print(f"waited {time.time() - t:.0f}s for the copy of {MODELS[key]['dir']}")
    TIMES[f"copy_{key}"] = COPY["seconds"].get(key, 0)

if not DRY_RUN:
    os.makedirs(LOCAL, exist_ok=True)
    threading.Thread(target=_copy_all, args=([MODEL, "l8", "q32"],), daemon=True).start()
    print(f"copying in the background: {MODELS[MODEL]['dir']} first, then Llama 8B and Qwen 32B")
import importlib.util
if importlib.util.find_spec("peft") is None:
    print("installing peft ...")
    !pip install -q peft
TRAIN = ["train", "--data_dir", TRAIN_DATA, "--out_dir", ADAPTER, "--merged_dir", MODELS[FT_KEY]["path"], "--n_train", N_TRAIN, "--n_val", N_VAL,
         "--ckpt_questions"] + CKPT_QUESTIONS + ["--save_every_questions", 1000, "--micro_batch", 4]
if DRY_RUN:
    run_stage(TRAIN + ["--backend", "tiny", "--n_train", 80, "--n_val", 6, "--grad_accum", 4, "--micro_batch", 2, "--max_len", 9000,
                       "--ckpt_questions", 20, 40, 80, "--save_every_questions", 16], "train.log")
elif not (trained() and merged()):
    wait_local(MODEL)
    run_stage(TRAIN + ["--base", MODELS[MODEL]["path"], "--time_limit_min", max(5.0, minutes_left() - 45)], "train.log")
TRAIN_DONE = trained() and (merged() or DRY_RUN)
print("\nTraining complete." if TRAIN_DONE else "\nTraining NOT COMPLETE YET - run this notebook again to continue from the last resume point.")

### 3. Retrieval on the 400 test questions, then answers (vLLM)

In [ ]:
BACKEND = "mock" if DRY_RUN else "vllm"
SETS = ["--old_sweep", OLD_SWEEP, "--old_data", OLD_DATA, "--fresh_run", FRESH_RUN, "--fresh_data", FRESH_DATA, "--run_dir", RUN_DIR,
        "--models_json", f"{RUN_DIR}/models.json", "--backend", BACKEND, "--variants", FT_NAME, "--add_variant", f"{FT_NAME}:{FT_KEY}:{FT_LABEL}"]
ALL_DONE = False
if TRAIN_DONE:
    if not DRY_RUN and importlib.util.find_spec("vllm") is None:
        t = time.time()
        !pip install -q vllm
        !pip uninstall -y -q torchaudio   # Colab's torchaudio is built for another CUDA version and breaks imports
        chk = subprocess.run([sys.executable, "-c", "from vllm import LLM; import vllm, torch; print('vLLM', vllm.__version__, '| torch', torch.__version__)"],
                             capture_output=True, text=True)
        print(chk.stdout or chk.stderr[-2000:])
        assert chk.returncode == 0, "vLLM does not import - Runtime -> Disconnect and delete runtime, then run again."
        TIMES["install_vllm"] = round(time.time() - t)
    run_stage(["retrieve"] + SETS, "retrieve.log")
    for key in ("l8", "q32"):
        wait_local(key)
    run_stage(["answer", "--cache_run", CACHE_RUN, "--cache_answers"] + CACHE_ANSWERS + SETS, "answer.log")
    ALL_DONE = True

### 4. Results

In [ ]:
if ALL_DONE:
    from IPython.display import Markdown, display
    run_stage(["evaluate", "--adapter_dir", ADAPTER] + SETS, "evaluate.log")
    display(Markdown(open(f"{RUN_DIR}/results/summary.md").read()))
    print("Excel workbook:", f"{RUN_DIR}/results/set7_retriever_finetune.xlsx")
TIMES["total_session"] = round(time.time() - T_START)
json.dump({"session": SESSION, "run_dir": RUN_DIR, "gpu": gpu, "seconds": TIMES, "code": EXPECTED_CODE, "complete": ALL_DONE},
          open(f"{SESSION_DIR}/session.json", "w"), indent=1)
print("Wall-clock this session (s):", TIMES)
print("\nALL DONE." if ALL_DONE else "\nNOT COMPLETE YET - run this notebook again to continue.")
try:
    drive.flush_and_unmount()
    print("Saved to Drive.")
except Exception as e:
    print("Drive flush skipped:", e)